In [2]:
# import packages and BTMS_model

import os
import sys
import io
import contextlib
import itertools
import numpy as np
import pandas as pd

from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

PROJECT_ROOT = os.path.abspath("../..")
sys.path.append(PROJECT_ROOT)

import lib.BTMS_model as BTMS_model
import CoolProp.CoolProp as CP

In [3]:
# result-file naming configuration
# Only define the result filename and exported Excel name.

CASE_ID = 'C0005'
RESULT_BASENAME = 'C0005R00_PARK25_NAC_1D_SCAN'
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'out', CASE_ID)
os.makedirs(OUTPUT_DIR, exist_ok=True)

RESULT_XLSX_NAME = f'{RESULT_BASENAME}.xlsx'

print(f'Result basename: {RESULT_BASENAME}')
print(f'Official Excel result: {RESULT_XLSX_NAME}')
print(f'Output directory: {OUTPUT_DIR}')


Result basename: C0005R00_PARK25_NAC_1D_SCAN
Official Excel result: C0005R00_PARK25_NAC_1D_SCAN.xlsx
Output directory: d:\Workspace\eBATS\out\C0005


In [4]:
# read Qbat(W) from the data file

# Keep the physical simulation duration unchanged, while using the original time step.
SIM_DURATION_S = 1920.0   # s, physical mission duration
dt = 1.0                 # s, integration time step

NUM_STEPS = int(round(SIM_DURATION_S / dt))
if not np.isclose(NUM_STEPS * dt, SIM_DURATION_S):
    raise ValueError('SIM_DURATION_S must be an integer multiple of dt.')

# Keep the original variable name for downstream compatibility.
# Here SIM_TIME_S means the number of numerical time steps, not the physical duration.
SIM_TIME_S = NUM_STEPS

file_name = os.path.join(
    PROJECT_ROOT,
    'data',
    'MD05E070207A1  data_power gen_single cell_Liu_20260421.xlsx'
)

df = pd.read_excel(file_name, sheet_name='Sheet1')
power_generation_data_1s = df['Qbat(W)'].dropna().to_numpy(dtype=float)

# The source Qbat data are treated as 1-s samples. For dt = 1.0 s,
# each 1-s heat-generation value is used for one numerical time step.
time_s = np.arange(SIM_TIME_S + 1) * dt
power_indices = np.floor(time_s[:-1]).astype(int)
power_indices = np.clip(power_indices, 0, len(power_generation_data_1s) - 1)
power_generation_data = power_generation_data_1s[power_indices]

print(f'Simulation duration: {SIM_DURATION_S:.1f} s')
print(f'Time step dt: {dt:.3f} s')
print(f'Number of numerical steps: {SIM_TIME_S}')
print(f'Q_gen array length: {len(power_generation_data)}')


Simulation duration: 1920.0 s
Time step dt: 1.000 s
Number of numerical steps: 1920
Q_gen array length: 1920


In [5]:
# model settings and natural-air-cooling scan ranges
#
# Maintenance note:
# In most cases, only edit the dictionaries/lists in this cell.
# The original variable names are kept below so that the downstream
# calculation logic and notebook structure do not need to be changed.

# battery cell properties
# Initial battery temperature is set equal to the scanned inlet air temperature in each case.
BATTERY_PROPS = {
    'm_bat': 48e-3,          # kg
    'cp_bat': 830.0,         # J/(kg K)
    'D_bat': 18e-3,          # m
    'H_bat': 65e-3,          # m
}

# 1D module layout
MODULE_LAYOUT = {
    'N_r': 16,               # number of cells in the transverse direction
    'N_c': 20,               # number of cells along the air-flow direction
    'num_cell_seg': 1.0,     # number of cells represented by each battery segment
}

# coolant / natural-air settings
AIR_SETTINGS = {
    'fluid': 'Air',
    'p_air': 101325.0,       # Pa
}

# natural-air-cooling operating period
# 0-180 s: no air cooling
# 180-1560 s: air cooling
# 1560-1920 s: no air cooling
COOLING_PERIOD = {
    't_cruise_start': 180.0,
    't_cruise_end': 1560.0,
}

# solver settings passed to BTMS_model.solve_coolant_temperature_distribution
SOLVER_SETTINGS = {
    'solver_tol': 1e-6,
    'solver_maxiter': 1000,
}

# mission-stage definitions for table indicators
mission_stages = [
    ('takeoff',       0.0,    6.0),
    ('climb',         6.0,   36.0),
    ('transition1',  36.0,  180.0),
    ('cruise',      180.0, 1560.0),
    ('transition2',1560.0, 1704.0),
    ('descent',    1704.0, 1734.0),
    ('hover',      1734.0, 1914.0),
    ('landing',    1914.0, 1920.0),
]

# parameter-scan ranges
# Natural wind speed is fixed at 1.5 m/s. The scan only varies T_air_in and s_gap.
SCAN_VALUES = {
    'V_air_list': np.array([1.5], dtype=float),                         # m/s, fixed natural wind speed
    'T_air_list': np.linspace(20.0, 40.0, 5),                            # degC
    's_gap_mm_list': np.linspace(1.0, 5.0, 5),                            # mm
}

# Keep original variable names for the rest of the notebook.
m_bat = BATTERY_PROPS['m_bat']
cp_bat = BATTERY_PROPS['cp_bat']
D_bat = BATTERY_PROPS['D_bat']
H_bat = BATTERY_PROPS['H_bat']
A_battery = np.pi * D_bat * H_bat

N_r = MODULE_LAYOUT['N_r']
N_c = MODULE_LAYOUT['N_c']
num_seg = N_c
num_cell_seg = MODULE_LAYOUT['num_cell_seg']

fluid = AIR_SETTINGS['fluid']
p_air = AIR_SETTINGS['p_air']

t_cruise_start = COOLING_PERIOD['t_cruise_start']
t_cruise_end = COOLING_PERIOD['t_cruise_end']

solver_tol = SOLVER_SETTINGS['solver_tol']
solver_maxiter = SOLVER_SETTINGS['solver_maxiter']

V_air_list = SCAN_VALUES['V_air_list']
T_air_list = SCAN_VALUES['T_air_list']
s_gap_mm_list = SCAN_VALUES['s_gap_mm_list']


In [6]:
# build natural-air-cooling scan cases

def build_scan_cases(V_air_list, T_air_list, s_gap_mm_list):
    """Build all combinations with fixed V_air = 1.5 m/s and assign subcase IDs under CASE_ID."""
    cases = []

    for case_count, (V_air, T_air, s_gap_mm) in enumerate(
        itertools.product(V_air_list, T_air_list, s_gap_mm_list),
        start=1
    ):
        cases.append({
            'case_id': f'{CASE_ID}_S{case_count:03d}',
            'V_air': float(V_air),
            'T_air': float(T_air),
            's_gap_mm': float(s_gap_mm),
        })

    return cases, pd.DataFrame(cases)


scan_cases, scan_cases_df = build_scan_cases(V_air_list, T_air_list, s_gap_mm_list)

print(f'Total cases: {len(scan_cases)}')
scan_cases_df.head()


Total cases: 25


,case_id,V_air,T_air,s_gap_mm
0,C0005_S001,1.5,20.0,1.0
1,C0005_S002,1.5,20.0,2.0
2,C0005_S003,1.5,20.0,3.0
3,C0005_S004,1.5,20.0,4.0
4,C0005_S005,1.5,20.0,5.0


In [7]:
# calculate air properties once for each inlet air temperature

air_props_cache = {}

for T_air in sorted(scan_cases_df['T_air'].unique()):
    T_air = float(T_air)
    T_air_K = T_air + 273.15

    mu_air = CP.PropsSI('V', 'T', T_air_K, 'P', p_air, fluid)
    rho_air = CP.PropsSI('D', 'T', T_air_K, 'P', p_air, fluid)
    cp_air = CP.PropsSI('C', 'T', T_air_K, 'P', p_air, fluid)
    k_air = CP.PropsSI('L', 'T', T_air_K, 'P', p_air, fluid)
    Pr_air = cp_air * mu_air / k_air

    air_props_cache[T_air] = {
        'mu_air': mu_air,
        'rho_air': rho_air,
        'cp_air': cp_air,
        'k_air': k_air,
        'Pr_air': Pr_air,
    }

print(f'Air properties calculated for {len(air_props_cache)} inlet temperatures.')

Air properties calculated for 5 inlet temperatures.


In [8]:
# run one parameter-scan case by calling BTMS_model

def run_one_case(case_id, V_air, T_air, s_gap_mm):
    air_props = air_props_cache[float(T_air)]

    # geometry derived from scanned cell gap
    S_T = D_bat + s_gap_mm * 1e-3
    W_module = D_bat + (N_r - 1) * S_T
    A_air = W_module * H_bat
    A_cool = A_air / N_r

    # heat-transfer coefficient from BTMS_model correlations
    Re_air = air_props['rho_air'] * V_air * D_bat / air_props['mu_air']
    eps_air = BTMS_model.select_correction_factor(no_col=N_c, Re=Re_air)
    Nu_air = BTMS_model.zukauskas_nusellet(
        Re_air,
        air_props['Pr_air'],
        eps_air,
        air_props['Pr_air']
    )
    h_air = Nu_air * air_props['k_air'] / D_bat
    htc_cool = np.ones(num_seg) * h_air

    # Initial battery temperature is set equal to the inlet air temperature for each case.
    T_bat = np.ones(num_seg) * T_air
    T_cool = np.ones(num_seg) * T_air

    T_bat_history = np.zeros((SIM_TIME_S + 1, num_seg))
    T_air_history = np.zeros((SIM_TIME_S + 1, num_seg))
    T_bat_history[0, :] = T_bat
    T_air_history[0, :] = T_cool

    for k in range(SIM_TIME_S):
        t = time_s[k]
        is_cool = (t >= t_cruise_start) and (t < t_cruise_end)

        # One battery node represents one cell in the natural-air row model.
        args = {
            'num_seg': num_seg,
            'num_seg_bat': num_seg,
            'dt': dt,
            'T_cool_pre': T_cool,
            'T_bat_pre': T_bat,
            'u_cool_in': V_air,
            'p_cool': p_air,
            'fluid_cool': fluid,
            'A_HT_seg': A_battery,
            'A_cool_cs': A_cool,
            'm_bat': m_bat,
            'cp_bat': cp_bat,
            'D_bat': D_bat,
            'T_cool_in': T_air,
            'T_cool_out': max(float(T_cool[-1]), T_air),
            'is_cool': is_cool,
            'htc_cool': htc_cool,
            'cp_cool': air_props['cp_air'],
            'rho_cool': air_props['rho_air'],
            'Q_gen': power_generation_data[k],
            'num_cell_seg': num_cell_seg,
            'debug': False,
        }

        try:
            with contextlib.redirect_stdout(io.StringIO()):
                T_dist = BTMS_model.solve_coolant_temperature_distribution(
                    args,
                    tol=solver_tol,
                    maxiter=solver_maxiter,
                    debug=False
                )
        except Exception as e:
            print("\nSolver failed inside run_one_case.")
            print(f"case_id = {case_id}")
            print(f"V_air = {V_air} m/s")
            print(f"T_air_in = {T_air} degC")
            print(f"s_gap = {s_gap_mm} mm")
            print(f"time step k = {k}")
            print(f"t = {t:.1f} s")
            print(f"is_cool = {is_cool}")
            print(f"Q_gen = {power_generation_data[k]} W")
            print(f"Re_air = {Re_air}")
            print(f"h_air = {h_air}")
            print(f"A_air = {A_air}")
            print(f"A_cool = {A_cool}")
            print(f"T_bat_min/max before solve = {np.min(T_bat)}, {np.max(T_bat)}")
            print(f"T_cool_min/max before solve = {np.min(T_cool)}, {np.max(T_cool)}")
            print(f"solver_tol = {solver_tol}")
            print(f"solver_maxiter = {solver_maxiter}")
            print(f"error = {repr(e)}")
            raise

        if not np.all(np.isfinite(T_dist)):
            print("\nSolver returned non-finite values.")
            print(f"case_id = {case_id}")
            print(f"V_air = {V_air} m/s")
            print(f"T_air_in = {T_air} degC")
            print(f"s_gap = {s_gap_mm} mm")
            print(f"time step k = {k}")
            print(f"t = {t:.1f} s")
            print(f"T_dist_min/max = {np.nanmin(T_dist)}, {np.nanmax(T_dist)}")
            raise FloatingPointError("Non-finite values detected in T_dist")

        T_cool = T_dist[:num_seg]
        T_bat = T_dist[num_seg:]

        T_bat_history[k + 1, :] = T_bat
        T_air_history[k + 1, :] = T_cool

    Tmax = np.max(T_bat_history, axis=1)
    Tmin = np.min(T_bat_history, axis=1)
    DeltaT = Tmax - Tmin
    Tair_out = T_air_history[:, -1]

    def idx(t):
        return int(round(t / dt))

    row = {
        'case_id': case_id,
        'V_air_m_s': V_air,
        'T_air_in_C': T_air,
        's_gap_mm': s_gap_mm,
        'S_T_mm': S_T * 1e3,
        'Tmax_mission_C': float(np.max(Tmax)),
        'DeltaT_mission_max_C': float(np.max(DeltaT)),
        'cruise_recovery_Tmax_C': float(Tmax[idx(t_cruise_start)] - Tmax[idx(t_cruise_end)]),
        'Tair_out_cruise_end_C': float(Tair_out[idx(t_cruise_end)]),
    }

    for stage_name, t0, t1 in mission_stages:
        row[f'dTmax_{stage_name}_C'] = float(Tmax[idx(t1)] - Tmax[idx(t0)])

    return row

In [ ]:
# run all cases and export the final xlsx table

import shutil


def format_excel_table(xlsx_path):
    """Apply the original Times New Roman table style to the exported Excel file."""
    wb = load_workbook(xlsx_path)
    ws = wb.active
    ws.title = 'NAC scan'

    body_font = Font(name='Times New Roman', size=10)
    header_font = Font(name='Times New Roman', size=10, bold=True)
    alignment = Alignment(horizontal='center', vertical='center')
    thin = Side(style='thin')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    for row in ws.iter_rows():
        for cell in row:
            cell.font = header_font if cell.row == 1 else body_font
            cell.alignment = alignment
            cell.border = border
            if cell.row > 1 and isinstance(cell.value, float):
                cell.number_format = '0.0000'

    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = ws.dimensions

    for col_idx, column_cells in enumerate(ws.columns, start=1):
        max_len = max(len(str(cell.value)) if cell.value is not None else 0 for cell in column_cells)
        ws.column_dimensions[get_column_letter(col_idx)].width = min(max(max_len + 2, 12), 28)

    wb.save(xlsx_path)

summary_rows = []

for n, case in enumerate(scan_cases, start=1):
    print(
        f"Running {n}/{len(scan_cases)}: {case['case_id']}, "
        f"V_air={case['V_air']}, "
        f"T_air={case['T_air']}, "
        f"s_gap={case['s_gap_mm']}"
    )

    try:
        summary_rows.append(run_one_case(**case))
    except Exception as e:
        print("\nParameter scan stopped because one case failed.")
        print(f"failed index = {n}/{len(scan_cases)}")
        print(f"failed case_id = {case['case_id']}")
        print(f"failed V_air = {case['V_air']} m/s")
        print(f"failed T_air_in = {case['T_air']} degC")
        print(f"failed s_gap = {case['s_gap_mm']} mm")
        print(f"error = {repr(e)}")
        raise

parameter_scan_table = pd.DataFrame(summary_rows)

column_order = [
    'case_id',
    'V_air_m_s',
    'T_air_in_C',
    's_gap_mm',
    'S_T_mm',
    'Tmax_mission_C',
    'DeltaT_mission_max_C',
    'cruise_recovery_Tmax_C',
    'Tair_out_cruise_end_C',
    'dTmax_takeoff_C',
    'dTmax_climb_C',
    'dTmax_transition1_C',
    'dTmax_cruise_C',
    'dTmax_transition2_C',
    'dTmax_descent_C',
    'dTmax_hover_C',
    'dTmax_landing_C',
]

parameter_scan_table = parameter_scan_table[column_order]

output_dir = OUTPUT_DIR
os.makedirs(output_dir, exist_ok=True)

xlsx_path = os.path.join(output_dir, RESULT_XLSX_NAME)

parameter_scan_table.to_excel(xlsx_path, index=False)
format_excel_table(xlsx_path)

print(f'Saved official Excel: {xlsx_path}')
print('Only one table file is generated for this case.')

parameter_scan_table.head()

# copy the scan result into the data directory for further reference
data_dir = os.path.join(PROJECT_ROOT, 'data')
os.makedirs(data_dir, exist_ok=True)
shutil.copy(xlsx_path, data_dir)
print(f'Copied scan result to data directory: {data_dir}')


Running 1/25: C0005_S001, V_air=1.5, T_air=20.0, s_gap=1.0


KeyboardInterrupt: 